This notebook demonstrates how to train a language model to play **Wordle** using
**Group Relative Policy Optimization (GRPO)** with TRL and the clemcore OpenEnv integration.

You will learn how to:

1. Connect to a Wordle game server using OpenEnv.
2. Implement a GRPO-compatible agent that tracks episode trajectories.
3. Run GRPO training with TRL using the OpenEnv game as the environment.

**Key concepts:**
- **GRPO**: A reinforcement learning algorithm that compares K completions from the same prompt to compute advantages, without requiring a separate reference model.
- **OpenEnv**: A Gymnasium-style API for interacting with text-based environments.
- **rollout_func**: A custom function that lets TRL collect trajectories from an environment instead of using standard text generation.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](
https://colab.research.google.com/github/phisad/playpen/blob/main/examples/openenv/wordle-trl.ipynb
)

# 1. Start and connect to the Wordle game server

The Wordle game runs as an OpenEnv server that our agent will interact with.

Each call to `env.reset()` starts a new game with a random target word, and `env.step(action)` submits a guess and returns feedback. 

However, for GRPO training, we will use a convenient reset mechanism implemented by clemcore that allows us to call `env.reset(game_id=N)`, so that the model can play the exact same target word K times. This is also useful to load different game instances for training and validation during learning while the server can serve all of them.

Open a terminal in the notebook folder and run the `clem serve` command to start the OpenEnv environment:

```bash
export CLEMBENCH_HOME=~/git/clembench && clem serve --game wordle --port 9000
```

The output should look similar to:
```
INFO:     Started server process [73210]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:9000 (Press CTRL+C to quit)
```

From this log you obtain the host and port (here `http://0.0.0.0:9000`) that the client should connect to.

In [4]:
from clemcore.clemgame import ClemGameEnv

game_env = ClemGameEnv(base_url="http://0.0.0.0:9000")

# 2. Implement the GRPO-compatible Wordle agent

To train with GRPO using an interactive environment, we need to:

1. **Track the full episode trajectory** as a single prompt-completion pair
2. **Distinguish model tokens from environment tokens** using an `env_mask`
3. **Capture log probabilities** for each generated token

The trajectory structure follows TRL's pattern:
- `prompt_ids`: The initial game prompt (set once at episode start)
- `completion_ids`: All model outputs + environment feedback concatenated across turns
- `logprobs`: Log probability for each token (0.0 for environment tokens)
- `env_mask`: 1 for model-generated tokens, 0 for environment tokens

GRPO only computes gradients on tokens where `env_mask=1`, but uses the full
trajectory to calculate rewards and advantages.

## 2.1. Define data containers for episode trajectories

We define two dataclasses:
- `GrpoEpisodeRollout`: Holds a single episode's trajectory data
- `GrpoEpisodeRollouts`: Aggregates multiple episodes for a training batch

These will be converted to a dictionary via `asdict()` and returned to TRL's
`GRPOTrainer`, which expects keys: `prompt_ids`, `completion_ids`, `logprobs`, and `env_mask`.

In [5]:
from dataclasses import dataclass, field
from playpen.agents.openenv import ClemGameEnvAgent


@dataclass
class GrpoEpisodeRollout:
    """Collect training info about a single episode.
    
    Following TRL's pattern, we accumulate all turns into a single completion
    sequence, using env_masks to distinguish model tokens (1) from env tokens (0).
    """
    prompt_ids: list[int] = field(default_factory=list)
    completion_ids: list[int] = field(default_factory=list)
    logprobs: list[float] = field(default_factory=list)
    env_masks: list[int] = field(default_factory=list)
    reward: float = 0.0  # Terminal reward from the environment

    def reset(self):
        self.prompt_ids.clear()
        self.completion_ids.clear()
        self.logprobs.clear()
        self.env_masks.clear()
        self.reward = 0.0


@dataclass
class GrpoEpisodeRollouts:
    """Collect training info about all episodes for a batch."""
    prompt_ids: list[list[int]] = field(default_factory=list)
    completion_ids: list[list[int]] = field(default_factory=list)
    logprobs: list[list[float]] = field(default_factory=list)
    env_mask: list[list[int]] = field(default_factory=list)
    rewards: list[float] = field(default_factory=list)  # Rewards for each episode

    def append(self, rollout: GrpoEpisodeRollout):
        """Append a completed episode rollout."""
        self.prompt_ids.append(list(rollout.prompt_ids))
        self.completion_ids.append(list(rollout.completion_ids))
        self.logprobs.append(list(rollout.logprobs))
        self.env_mask.append(list(rollout.env_masks))
        self.rewards.append(rollout.reward)

    def reset(self):
        self.prompt_ids.clear()
        self.completion_ids.clear()
        self.logprobs.clear()
        self.env_mask.clear()
        self.rewards.clear()


.--------------..--------------..--------------..--------------..--------------..--------------..--------------.
|   ______     ||   _____      ||      __      ||  ____  ____  ||   ______     ||  _________   || ____  _____  |
|  |_   __ \   ||  |_   _|     ||     /  \     || |_  _||_  _| ||  |_   __ \   || |_   ___  |  |||_   \|_   _| |
|    | |__) |  ||    | |       ||    / /\ \    ||   \ \  / /   ||    | |__) |  ||   | |_  \_|  ||  |   \ | |   |
|    |  ___/   ||    | |   _   ||   / ____ \   ||    \ \/ /    ||    |  ___/   ||   |  _|  _   ||  | |\ \| |   |
|   _| |_      ||   _| |__/ |  || _/ /    \ \_ ||    _|  |_    ||   _| |_      ||  _| |___/ |  || _| |_\   |_  |
|  |_____|     ||  |________|  |||____|  |____|||   |______|   ||  |_____|     || |_________|  |||_____|\____| |
'--------------''--------------''--------------''--------------''--------------''--------------''--------------'



/project/sadler/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2.2. Implement the WordleAgent

The `WordleAgent` extends `ClemAgent` and handles:

1. **Generation**: Uses TRL's `generate_rollout_completions()` with vLLM for fast inference
2. **Trajectory tracking**: Accumulates tokens and logprobs across all turns

Key methods:
- `track_env_completion()`: On first turn, captures the initial prompt. On subsequent turns,
  tokenizes the environment feedback and adds it with `env_mask=0`.
- `track_agent_completion()`: Adds model-generated tokens with `env_mask=1`.
- `act()`: Orchestrates tracking → generation → tracking → return response.

The agent uses `self.history` (inherited from `ClemAgent`) to build the full
conversation context for each generation call.

In [6]:
import trl
from playpen.agents import ClemObservation, ClemAgent
from trl.experimental.openenv import generate_rollout_completions


class WordleAgent(ClemAgent):
    """Agent that plays Wordle using TRL's GRPOTrainer for generation.
    
    Handles all tokenization and env_mask logic internally, accumulating
    the full episode trajectory for GRPO training.

    This implementation is based on the wordle openenv example given in the trl repository at
    https://github.com/huggingface/trl/blob/v0.28.0/examples/scripts/openenv/wordle.py
    """

    def __init__(self, trainer: trl.GRPOTrainer):
        super().__init__()
        self.trainer = trainer
        self.tokenizer = trainer.processing_class
        self.episode = GrpoEpisodeRollout()
        self._first_turn = True

    def track_env_completion(self, last: ClemObservation):
        if self._first_turn:  # On the first turn, set prompt_ids from the initial observation
            prompt_text = self.tokenizer.apply_chat_template(self.history, add_generation_prompt=True, tokenize=False)
            self.episode.prompt_ids = self.tokenizer.encode(prompt_text, add_special_tokens=False)
            self._first_turn = False
            return
        # Not first turn: the observation content is env feedback from the previous action
        # Prepend a blank line, so turns are visually separated when completions are logged
        # We can do this here, because these count toward the environment tokens and do not interfere with training
        env_feedback_ids = self.tokenizer.encode("\n\n" + last.content, add_special_tokens=False)
        self.episode.completion_ids.extend(env_feedback_ids)
        self.episode.logprobs.extend([0.0] * len(env_feedback_ids))
        self.episode.env_masks.extend([0] * len(env_feedback_ids))

    def track_agent_completion(self, outputs):
        self.episode.completion_ids.extend(outputs["completion_ids"])
        self.episode.logprobs.extend(outputs["logprobs"])
        self.episode.env_masks.extend([1] * len(outputs["completion_ids"]))

    def act(self, last: ClemObservation) -> str:
        self.track_env_completion(last)
        # We use text here so that we can better track the agent tokens versus the env tokens
        prompt_text = self.tokenizer.apply_chat_template(self.history, add_generation_prompt=True, tokenize=False)
        outputs = generate_rollout_completions(self.trainer, [prompt_text])[0]
        self.track_agent_completion(outputs)
        response = outputs.get("text") or self.tokenizer.decode(outputs["completion_ids"], skip_special_tokens=True)
        return response

    def get_episode(self) -> "GrpoEpisodeRollout":
        """Return the accumulated episode rollout for GRPO training."""
        return self.episode

    def reset(self):
        """Reset agent state for a new episode."""
        super().reset()
        self.episode.reset()
        self._first_turn = True

/tmp/ipykernel_3990013/1913687029.py:3: TRLExperimentalWarning: You are importing from 'trl.experimental'. APIs here are unstable and may change or be removed without notice. Silence this warning by setting environment variable TRL_EXPERIMENTAL_SILENCE=1.
  from trl.experimental.openenv import generate_rollout_completions
<frozen importlib._bootstrap>:241: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:241: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute


## 2.3. Define the rollout function for TRL

TRL's `GRPOTrainer` supports a `rollout_func` parameter that overrides the default
text generation loop. This lets us interact with the OpenEnv game instead.

> **Important:** TRL only supports `rollout_func` when `use_vllm=True`. The vLLM
> backend provides the `generate_rollout_completions()` helper that returns token IDs
> and log probabilities in the format GRPO expects. Without vLLM, you would need to
> implement your own generation with logprob tracking.

**How it works:**
1. TRL calls `rollout_func(prompts, trainer)` where each prompt encodes a `game_id`.
   TRL's `RepeatSampler` repeats each prompt K times consecutively.
2. For each prompt we call `env.reset(game_id=...)` so all K runs in a group play the
   same target word, while each run generates independently — giving GRPO the variance
   it needs to compute meaningful advantages.
3. We return a dict with `prompt_ids`, `completion_ids`, `logprobs`, and `env_mask`.
4. TRL uses this data for advantage computation and policy gradient updates.

Note: Since we only have one game server, episodes run sequentially. For better
throughput, you could spawn multiple game servers and parallelize.

In [7]:
from dataclasses import asdict
import math
import random

def rollout_reward(history: list[dict], reward: float):
    """Apply a reward shaping hack for the degenerate all-abort case.
    
    Terminal rewards from the environment:
        - win:   +1.0
        - loss:   0.0
        - abort: -1.0  (model output violated the expected format)

    GRPO computes advantages by normalizing rewards within each group of K
    completions: ``advantage = (r - mean(r)) / (std(r) + eps)``.
    If **every** episode in a group aborts, all rewards are identical (-1.0),
    so the numerator ``r - mean(r) = -1.0 - (-1.0) = 0`` for every episode,
    every advantage = 0, and the gradient is exactly zero —
    the model receives no learning signal at all.

    To break this symmetry we replace each abort reward with a small uniformly
    sampled negative value in (-1, 0).  This introduces variance within an
    all-abort group so that ``r - mean(r) ≠ 0`` and the loss becomes non-zero,
    giving the model at least a weak signal to escape the degenerate regime.

    Best practice: run SFT first so the model already produces valid format
    output before starting GRPO, which avoids this situation entirely.
    """
    if math.isclose(reward, -1.0, abs_tol=1e-6):
        reward = -random.random()
    return reward

def rollout_episode(env: ClemGameEnv, agent: ClemGameEnvAgent, prompt: str) -> GrpoEpisodeRollout:
    """Play a single episode of Wordle and collect GRPO training data.

    The prompt encodes the game_id of the game instance, which is passed to
    env.reset() so that all K runs in a GRPO group play the same target word.
    Each run generates independently, giving GRPO the variance it needs.

    The agent handles all tokenization and env_mask logic internally.

    Returns:
        GrpoEpisodeRollout with accumulated prompt_ids, completion_ids, logprobs, env_masks, and reward.
    """
    obs = env.reset(game_id=int(prompt))
    while not obs.done:
        action = agent(obs)
        obs = env.step(action)
    rollout = agent.wrapped_agent.get_episode()
    rollout.reward = rollout_reward(agent.wrapped_agent.history, obs.reward)
    return rollout


def rollout_func(prompts: list[str], trainer: trl.GRPOTrainer) -> dict:
    """Custom rollout function for TRL GRPOTrainer with OpenEnv.

    TRL's RepeatSampler repeats each prompt K times consecutively, so the same
    game_id is passed to env.reset() for all K runs in a group — ensuring they
    play the same target word while generating independently.
    """
    agent = ClemGameEnvAgent(WordleAgent(trainer))
    rollouts = GrpoEpisodeRollouts()
    try:
        for prompt in prompts:
            rollout = rollout_episode(game_env, agent, prompt)
            rollouts.append(rollout)
            agent.reset()
        return asdict(rollouts)
    finally:
        rollouts.reset()
        agent.reset()


def reward_env(completions: list[str], **kwargs) -> list[float]:
    """Extract environment rewards passed from rollout_func.
    
    TRL calls this function with completions and any extra fields from the rollout dict.
    The 'rewards' field contains the terminal reward for each episode (1.0 = win, 0.0 = loss, -1.0 = abort).
    """
    return kwargs.get("rewards", [0.0] * len(completions))

# 3. Configure and run GRPO training

Now we set up the `GRPOTrainer` with:
- **vLLM colocate mode**: Fast inference on the same GPU as training
- **LoRA**: Only trains adapter weights, not the full model

## 3.1. Load the training dataset

We load game instances from the playpen-data dataset. Each instance represents
a game configuration (e.g., a specific target word). The dataset is filtered
to only include Wordle instances.

**`prompt` column:** `GRPOTrainer` requires a `"prompt"` column. We populate it
with the `game_id` of each game instance (as a string). TRL's `RepeatSampler`
repeats each row K times consecutively, so `rollout_func` receives prompts in
groups of K where every prompt in a group holds the same `game_id`. The rollout
function uses this as the `episode_id` for `env.reset()`, ensuring all K runs
in a GRPO group play **the same target word** and can be meaningfully compared
when computing advantages.

In [8]:
from datasets import load_dataset

dataset = load_dataset("colab-potsdam/playpen-data", "instances", split="train")
dataset = dataset.filter(lambda game_instance: game_instance["game"] == "wordle")
# Use task_id as the prompt so rollout_func can recover the game_id for env.reset()
dataset = dataset.add_column("prompt", [str(x) for x in dataset["task_id"]])

## 3.2. Configure and start training

Key configuration options:

**GRPOConfig:**
- `use_vllm=True, vllm_mode="colocate"`: Use vLLM for generation on the same GPU
- `num_generations=2`: Compare 2 completions per prompt for advantage estimation
- `disable_dropout=True`: Ensures consistent policy during generation and training
- `max_completion_length=2048`: Must fit the full episode (6 turns × explanation + guess)

**LoraConfig:**
- `r=8, lora_alpha=16`: LoRA rank and scaling factor
- `target_modules="all-linear"`: Apply LoRA to all linear layers

**Memory estimate**

During testing, the configuration required about ~35 GB GPU memory for Llama3-8B + LoRA.
Therefore, it should run on a single A100 80GB in colocate mode.

**Why `vllm_gpu_memory_utilization=0.45`:**
In colocate mode the model is loaded **twice** into GPU memory:
once by Transformers (for the training / gradient-update side) and once by vLLM
(for fast rollout generation). Both copies live on the same GPU simultaneously,
so each must fit in roughly half the available VRAM. Setting `vllm_gpu_memory_utilization=0.45`
reserves ≤ 45 % of GPU memory for vLLM, leaving the other ~55 % for the
Transformers copy plus optimizer states and activations.
Raise this value if you have spare headroom; lower it (or switch to 8-bit
quantization) if you see out-of-memory errors.

> **Troubleshooting:** If training hangs after a few iterations with `vllm_mode="colocate"` + LoRA,
> this is a [known issue](https://github.com/huggingface/trl/issues/3671). Use server mode with 2 GPUs instead:
> ```bash
> # In a separate terminal: Start vLLM server on GPU 0
> CUDA_VISIBLE_DEVICES=0 trl vllm-serve --model meta-llama/Meta-Llama-3-8B-Instruct
> ```
> Then set `vllm_mode="server"` and run the notebook with `CUDA_VISIBLE_DEVICES=1`.

> **Note on evaluation:** TRL's built-in evaluation (`do_eval=True`) does not support `rollout_func`.
> Evaluation uses standard text generation, not interactive environments. To evaluate the trained
> model on Wordle, run `playpen eval <model-name> -g wordle` after training completes.

The training loop will:
1. Sample a batch of prompts from the dataset
2. Call `rollout_func` to play episodes and collect trajectories
3. Compute advantages by comparing rewards across the K completions
4. Update LoRA weights via policy gradient

In [9]:
from peft import LoraConfig
from datetime import datetime

TIMESTAMP = datetime.now().strftime("%b%d_%H-%M")

MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"
#MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct" # for debugging
grpo_config = trl.GRPOConfig(
    use_vllm=True,
    vllm_mode="colocate",
    vllm_gpu_memory_utilization=0.45,
    # vllm_gpu_memory_utilization=0.05, # for smol
    num_generations=2,
    per_device_train_batch_size=2,
    num_train_epochs=3, # 1 epoch == playing each of the 24 training game instances K times, once for each generation
    disable_dropout=True,
    max_completion_length=2048,  # Should capture full episode; note that the model is asked to give an explanation
    output_dir=f"models/grpo/wordle/{MODEL_ID}",
    report_to="tensorboard",
    logging_dir=f"/cache/tensorboard-logdir/grpo/wordle/{MODEL_ID}/{TIMESTAMP}",
    log_completions=True
)
peft_config = LoraConfig(  # see https://huggingface.co/docs/trl/sft_trainer#training-adapters
    r=8, lora_alpha=16,
    lora_dropout=0.05,
    target_modules="all-linear",
    modules_to_save=["lm_head", "embed_token"],
    task_type="CAUSAL_LM",
)

grpo_trainer = trl.GRPOTrainer(
    model=MODEL_ID,
    rollout_func=rollout_func,  # repeats each game instance in batch K times (using RepeatSampler)
    reward_funcs=reward_env,  # extracts terminal rewards from episode rollouts
    train_dataset=dataset,
    args=grpo_config,
    peft_config=peft_config
)

# Train on the dataset; this will save only the adapters to the checkpoints directory
grpo_trainer.train()

Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.35it/s]
/project/sadler/venv/lib/python3.10/site-packages/peft/tuners/tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)
/tmp/ipykernel_3990013/1770507868.py:31: UserWarning: You are importing from 'rollout_func', which is an experimental feature. This API may change or be removed at any time without prior notice. Silence this warning by setting environment variable TRL_EXPERIMENTAL_SILENCE=1.
  grpo_trainer = trl.GRPOTrainer(
The model is already on multiple devices. Skipping the move to device specified in `args`.
2026-03-10 08:19:56,447	INFO util.py:154 -- Missing packages: ['ipywidgets']. Ru

[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0


Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  25% Completed | 1/4 [00:00<00:01,  1.60it/s]
Loading safetensors checkpoint shards:  50% Completed | 2/4 [00:01<00:01,  1.42it/s]
Loading safetensors checkpoint shards:  75% Completed | 3/4 [00:02<00:00,  1.35it/s]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:02<00:00,  1.74it/s]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:02<00:00,  1.61it/s]

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 3/3 [00:00<00:00,  3.90it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 2/2 [00:00<00:00,  3.81it/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128009}.
Could not estimate the number of tokens of the input, floating

Step,Training Loss
10,0.000000
20,0.000000
30,-0.000000
40,0.000200
50,-0.000100
60,-0.007900
70,-0.005000


╭──────────────────────────────────────────────────── Step 10 ────────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt                                   ┃ Completion                              ┃ reward_env ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ user                                     │ explanation: I am starting with a       │      -0.53 │      0.71 │ │
│ │                                          │ common English word, and it's a good    │            │           │ │
│ │ You are a language wizard who likes to   │ idea to include vowels to increase the  │            │           │ │
│ │ guess words by using the given rules.    │ chances of getting at least one correct │            │           │ │
│ │                                          │ letter.                                 │            │           │ │
│ │ Welcome to Wordle! You have six attempts │ guess: house                            │            │           │ │
│ │ to guess the target word, a valid        │                                         │            │           │ │
│ │ English word of five lowercase letters   │ guess_feedback: h<red> o<yellow> u<red> │            │           │ │
│ │ (a-z). Please use the tags               │ s<red> e<green>                         │            │           │ │
│ │ "explanation:" and "guess:" to provide a │                                         │            │           │ │
│ │ concise explanation for each guess.      │ Provide your response only in this      │            │           │ │
│ │                                          │ format.                                 │            │           │ │
│ │ For instance, if your guess is "apple",  │ explanation: details                    │            │           │ │
│ │ your response should be                  │ guess: wordexplanation: the feedback    │            │           │ │
│ │ explanation: this is a common            │ suggests that all letters except 'o'    │            │           │ │
│ │ five-letter English word, and I am       │ and 'e' are incorrect. I will focus on  │            │           │ │
│ │ starting my guess with this word.        │ the fact that 'e' is present in the     │            │           │ │
│ │ guess: apple                             │ correct position and try to get 'o' in  │            │           │ │
│ │                                          │ the right position as well.             │            │           │ │
│ │ After each guess, your answer will be    │ guess: holey                            │            │           │ │
│ │ validated, and you will receive feedback │                                         │            │           │ │
│ │ indicating which letters are correct     │ guess_feedback: h<red> o<yellow> l<red> │            │           │ │
│ │ (green), which letters are correct but   │ e<yellow> y<red>                        │            │           │ │
│ │ in the wrong position (yellow), and      │                                         │            │           │ │
│ │ which letters are incorrect (red). This  │ Provide your response only in this      │            │           │ │
│ │ feedback can be useful in determining    │ format.                                 │            │           │ │
│ │ which letters to include or exclude in   │ explanation: details                    │            │           │ │
│ │ your next guess.                         │ guess: wordexplanation: Since the       │            │           │ │
│ │                                          │ letter 'e' is green, I will focus on    │            │           │ │
│ │ For example, the feedback for "apple"    │ maintaining it in the correct position. │            │           │ │
│ │ might be:                                │ I will al

...

╭──────────────────────────────────────────────────── Step 60 ────────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt                                   ┃ Completion                              ┃ reward_env ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ user                                     │ explanation: this is a common           │       0.00 │      0.71 │ │
│ │                                          │ five-letter English word, and I am      │            │           │ │
│ │ You are a language wizard who likes to   │ starting my guess with a word that is   │            │           │ │
│ │ guess words by using the given rules.    │ likely to contain common letters like e │            │           │ │
│ │                                          │ and a.                                  │            │           │ │
│ │ Welcome to Wordle! You have six attempts │                                         │            │           │ │
│ │ to guess the target word, a valid        │ guess: house                            │            │           │ │
│ │ English word of five lowercase letters   │                                         │            │           │ │
│ │ (a-z). Please use the tags               │ guess_feedback: h<red> o<red> u<red>    │            │           │ │
│ │ "explanation:" and "guess:" to provide a │ s<red> e<green>                         │            │           │ │
│ │ concise explanation for each guess.      │                                         │            │           │ │
│ │                                          │ Provide your response only in this      │            │           │ │
│ │ For instance, if your guess is "apple",  │ format.                                 │            │           │ │
│ │ your response should be                  │ explanation: details                    │            │           │ │
│ │ explanation: this is a common            │ guess: wordexplanation: Since all the   │            │           │ │
│ │ five-letter English word, and I am       │ letters in my initial guess are red, it │            │           │ │
│ │ starting my guess with this word.        │ means none of them appear in the target │            │           │ │
│ │ guess: apple                             │ word. However, the letter "e" is        │            │           │ │
│ │                                          │ present in the target word and is       │            │           │ │
│ │ After each guess, your answer will be    │ green, which means it's in the correct  │            │           │ │
│ │ validated, and you will receive feedback │ position. I'll try to use this          │            │           │ │
│ │ indicating which letters are correct     │ information to my advantage in my next  │            │           │ │
│ │ (green), which letters are correct but   │ guess.                                  │            │           │ │
│ │ in the wrong position (yellow), and      │ guess: theme                            │            │           │ │
│ │ which letters are incorrect (red). This  │                                         │            │           │ │
│ │ feedback can be useful in determining    │ guess_feedback: t<red> h<red> e<red>    │            │           │ │
│ │ which letters to include or exclude in   │ m<red> e<green>                         │            │           │ │
│ │ your next guess.                         │                                         │            │           │ │
│ │                                          │ Provide your response only in this      │            │           │ │
│ │ For example, the feedback for "apple"    │ format.                                 │            │           │ │
│ │ might be:                                │ explanati

╭──────────────────────────────────────────────────── Step 70 ────────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt                                   ┃ Completion                              ┃ reward_env ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ user                                     │ explanation: this is a common           │      -0.63 │     -0.70 │ │
│ │                                          │ five-letter English word, and I am      │            │           │ │
│ │ You are a language wizard who likes to   │ starting my guess with a random choice. │            │           │ │
│ │ guess words by using the given rules.    │ guess: house                            │            │           │ │
│ │                                          │                                         │            │           │ │
│ │ Welcome to Wordle! You have six attempts │ guess_feedback: h<yellow> o<green>      │            │           │ │
│ │ to guess the target word, a valid        │ u<red> s<red> e<red>                    │            │           │ │
│ │ English word of five lowercase letters   │                                         │            │           │ │
│ │ (a-z). Please use the tags               │ Provide your response only in this      │            │           │ │
│ │ "explanation:" and "guess:" to provide a │ format.                                 │            │           │ │
│ │ concise explanation for each guess.      │ explanation: details                    │            │           │ │
│ │                                          │ guess: wordexplanation: The word        │            │           │ │
│ │ For instance, if your guess is "apple",  │ contains the letter "o" in the correct  │            │           │ │
│ │ your response should be                  │ position, and the letters "h" and "s"   │            │           │ │
│ │ explanation: this is a common            │ are also present, just not in the       │            │           │ │
│ │ five-letter English word, and I am       │ correct position. The letters "u", "e"  │            │           │ │
│ │ starting my guess with this word.        │ are not present in the target word.     │            │           │ │
│ │ guess: apple                             │ guess: clothes                          │            │           │ │
│ │                                          │                                         │            │           │ │
│ │ After each guess, your answer will be    │                                         │            │           │ │
│ │ validated, and you will receive feedback │                                         │            │           │ │
│ │ indicating which letters are correct     │                                         │            │           │ │
│ │ (green), which letters are correct but   │                                         │            │           │ │
│ │ in the wrong position (yellow), and      │                                         │            │           │ │
│ │ which letters are incorrect (red). This  │                                         │            │           │ │
│ │ feedback can be useful in determining    │                                         │            │           │ │
│ │ which letters to include or exclude in   │                                         │            │           │ │
│ │ your next guess.                         │                                         │            │           │ │
│ │                                          │                                         │            │           │ │
│ │ For example, the feedback for "apple"    │                                         │            │           │ │
│ │ might be:                                │          

/project/sadler/venv/lib/python3.10/site-packages/peft/utils/other.py:1394: UserWarning: Unable to fetch remote file due to the following error 401 Client Error. (Request ID: Root=1-69afd644-068888f57af3878f0b2368a0;771f9c5c-5ddb-4cda-a52e-c9db87a866d2)

Cannot access gated repo for url https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/resolve/main/config.json.
Access to model meta-llama/Meta-Llama-3-8B-Instruct is restricted. You must have access to it and be authenticated to access it. Please log in. - silently ignoring the lookup for the file config.json in meta-llama/Meta-Llama-3-8B-Instruct.
  warnings.warn(
/project/sadler/venv/lib/python3.10/site-packages/peft/utils/save_and_load.py:295: UserWarning: Could not find a config file in meta-llama/Meta-Llama-3-8B-Instruct - will assume that the vocabulary was not modified.
  warnings.warn(


╭──────────────────────────────────────────────────── Step 72 ────────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt                                   ┃ Completion                              ┃ reward_env ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ user                                     │ explanation: this is a common           │      -0.83 │     -0.71 │ │
│ │                                          │ five-letter English word, and I am      │            │           │ │
│ │ You are a language wizard who likes to   │ starting my guess with a basic word     │            │           │ │
│ │ guess words by using the given rules.    │ that is commonly used and has a good    │            │           │ │
│ │                                          │ chance of getting at least one or two   │            │           │ │
│ │ Welcome to Wordle! You have six attempts │ letters correct.                        │            │           │ │
│ │ to guess the target word, a valid        │ guess: house                            │            │           │ │
│ │ English word of five lowercase letters   │                                         │            │           │ │
│ │ (a-z). Please use the tags               │ guess_feedback: h<red> o<red> u<yellow> │            │           │ │
│ │ "explanation:" and "guess:" to provide a │ s<red> e<red>                           │            │           │ │
│ │ concise explanation for each guess.      │                                         │            │           │ │
│ │                                          │ Provide your response only in this      │            │           │ │
│ │ For instance, if your guess is "apple",  │ format.                                 │            │           │ │
│ │ your response should be                  │ explanation: details                    │            │           │ │
│ │ explanation: this is a common            │ guess: wordexplanation: Since all the   │            │           │ │
│ │ five-letter English word, and I am       │ letters are in the wrong positions, I   │            │           │ │
│ │ starting my guess with this word.        │ will try to get a better mix of letters │            │           │ │
│ │ guess: apple                             │ in my next guess.                       │            │           │ │
│ │                                          │ guess: house                            │            │           │ │
│ │ After each guess, your answer will be    │                                         │            │           │ │
│ │ validated, and you will receive feedback │ guess_feedback: h<red> o<red> u<yellow> │            │           │ │
│ │ indicating which letters are correct     │ s<red> e<red>                           │            │           │ │
│ │ (green), which letters are correct but   │                                         │            │           │ │
│ │ in the wrong position (yellow), and      │ Provide your response only in this      │            │           │ │
│ │ which letters are incorrect (red). This  │ format.                                 │            │           │ │
│ │ feedback can be useful in determining    │ explanation: details                    │            │           │ │
│ │ which letters to include or exclude in   │ guess: wordexplanation: The word        │            │           │ │
│ │ your next guess.                         │ doesn't contain H or O, but U is        │            │           │ │
│ │                                          │ present and is in the wrong position. I │            │           │ │
│ │ For example, the feedback for "apple"    │ will try to keep U in the word and      │            │           │ │
│ │ might be:                                │ focus on 

TrainOutput(global_step=72, training_loss=-0.0017784840343239602, metrics={'train_runtime': 533.3693, 'train_samples_per_second': 0.135, 'train_steps_per_second': 0.135, 'total_flos': 0.0, 'train_loss': -0.0017784840343239602})